# Funnel system — provenance-routed fake-news detection

**The system under test:** a RoBERTa AI-vs-Human **router** (§7m) sends each article either to
**paper-faithful LIFE** (BCE head, §7e — the specialist for LLM-generated news) or to a **plain BERT**
(§7o/§7q — the specialist for human news). The whole thing is scored as ONE binary fake-vs-real
classifier (fake = HF+MF = 1, real = HR+MR = 0) on a shared held-out test set.

**Why:** LIFE detects LLM-*generation*, not fakeness (§7j negative control), while a content BERT
does separate human fake from real (§7q). Routing each article to the expert that actually has
signal is the constructive synthesis — IF it beats doing nothing clever.

| variant | routing | machine branch | question it answers |
|---|---|---|---|
| funnel | RoBERTa router | LIFE | the system as proposed |
| oracle | true provenance | LIFE | what do the router's errors cost? |
| two_bert | RoBERTa router | plain BERT | does LIFE earn its branch in-system? |
| oracle_two_bert | true provenance | plain BERT | ceiling of the two-BERT variant |
| monolithic | none | — | does routing add anything at all? |

**The split** (`funnel_system/make_split.py`): one 70/15/15 train/val/test split per seed (7/42/123),
shared by every component. MF/MR are GPT-3.5 rewrites of HF/HR stories and share their ids, so the
split is made over **id-groups** — a story and all its rewrites land on the same side, otherwise every
veracity model would train on the content of test articles.

Run order: top to bottom. PolitiFact++ first (pilot, ~85-article test — plumbing validation only),
then GossipCop++ (~3,078-article test — the citable result).

In [ ]:
# GPU + deps (transformers, fastNLP etc. for the LIFE branch)
!nvidia-smi
%pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/LIFE'

DATASET_ROOT = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset'
POLITIFACT_DIR = f'{DATASET_ROOT}/PolitiFact++'
GOSSIPCOP_DIR = f'{DATASET_ROOT}/GossipCop++'

# LIFE features already on Drive (LLaMA2-7B reconstruction; see the inventory cell)
PF_FEATURES = f'{PROJECT_DIR}/dataset/features_llama_multi'            # all four subsets, 520 articles
GC_FEATURES_MULTI = f'{PROJECT_DIR}/dataset/gossipcop/features_llama_multi'  # all four (if the 4-class track ran)
GC_FEATURES_BIN = f'{PROJECT_DIR}/dataset/gossipcop/features_llama'          # MF/MR only (binary track)

# Everything this notebook produces lands under funnel_system/
PF_RUN = f'{PROJECT_DIR}/funnel_system/run_politifact'
GC_RUN = f'{PROJECT_DIR}/funnel_system/run_gossipcop'
GC_MISROUTE_RAW = f'{PROJECT_DIR}/funnel_system/gc_misroute_raw'
GC_MISROUTE_FEATURES = f'{PROJECT_DIR}/funnel_system/gc_misroute_features'
for d in (PF_RUN, GC_RUN):
    os.makedirs(d, exist_ok=True)

os.chdir(PROJECT_DIR)
print('cwd:', os.getcwd())

In [ ]:
# Feature inventory — decides the GossipCop LIFE coverage path.
# Expected if complete: PF_FEATURES ~ HF 97 / HR 194 / MF 97 / MR 132 (minus a few drops);
# GC_FEATURES_BIN ~ MF 4084 / MR 4169; GC_FEATURES_MULTI additionally HF 4084 / HR 8168.
import os

def inventory(d):
    if not os.path.isdir(d):
        print(f'MISSING   {d}')
        return
    print(f'exists    {d}')
    for fn in sorted(os.listdir(d)):
        if fn.endswith('.jsonl'):
            with open(os.path.join(d, fn), encoding='utf-8') as f:
                n = sum(1 for _ in f)
            print(f'          {fn}: {n} records')

for d in (PF_FEATURES, GC_FEATURES_MULTI, GC_FEATURES_BIN):
    inventory(d)

## PolitiFact++ — pilot (plumbing validation)

~85-article test sets: every number here is ±5pp noise. The pilot exists to prove the split/join/
prediction plumbing end to end before spending GossipCop compute — do NOT read conclusions from it.
LIFE features for all 520 articles already exist (`features_llama_multi`, reused by §7j), so this
section needs no feature generation.

In [ ]:
# Unified id-group split, seeds 7/42/123
!python funnel_system/make_split.py --data_dir "{POLITIFACT_DIR}" --out_dir "{PF_RUN}" --name PolitiFact++

In [ ]:
# Router (RoBERTa AI-vs-Human, §7m config)
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed7.json" --role router --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed42.json" --role router --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed123.json" --role router --name PolitiFact++ --out_dir "{PF_RUN}"

In [ ]:
# Human expert (plain BERT on HF/HR, §7o/§7q config, best-val)
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed7.json" --role human_expert --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed42.json" --role human_expert --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed123.json" --role human_expert --name PolitiFact++ --out_dir "{PF_RUN}"

In [ ]:
# Machine-branch BERT control (plain BERT on MF/MR)
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed7.json" --role machine_bert --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed42.json" --role machine_bert --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed123.json" --role machine_bert --name PolitiFact++ --out_dir "{PF_RUN}"

In [ ]:
# Monolithic control (one BERT on all four subsets, no routing)
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed7.json" --role monolithic --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed42.json" --role monolithic --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed123.json" --role monolithic --name PolitiFact++ --out_dir "{PF_RUN}"

In [ ]:
# Assemble LIFE train/test feature files for each seed's split
!python funnel_system/assemble_life_data.py --features_dir "{PF_FEATURES}" --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed7.json" --out_dir "{PF_RUN}"
!python funnel_system/assemble_life_data.py --features_dir "{PF_FEATURES}" --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed42.json" --out_dir "{PF_RUN}"
!python funnel_system/assemble_life_data.py --features_dir "{PF_FEATURES}" --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed123.json" --out_dir "{PF_RUN}"

In [ ]:
# LIFE branch (paper-faithful BCE head) per seed. Per-epoch printed metrics are over the
# FULL mixed test set - ignore them; the LIFE anchor comes from evaluate_system.py.
!python LIFE_train/train_bce.py --train_path "{PF_RUN}/life_train_seed7.jsonl" --test_path "{PF_RUN}/life_test_seed7.jsonl" --num_train_epochs 50 --seed 7 --pred_out "{PF_RUN}/life_preds_seed7.jsonl"
!python LIFE_train/train_bce.py --train_path "{PF_RUN}/life_train_seed42.jsonl" --test_path "{PF_RUN}/life_test_seed42.jsonl" --num_train_epochs 50 --seed 42 --pred_out "{PF_RUN}/life_preds_seed42.jsonl"
!python LIFE_train/train_bce.py --train_path "{PF_RUN}/life_train_seed123.jsonl" --test_path "{PF_RUN}/life_test_seed123.jsonl" --num_train_epochs 50 --seed 123 --pred_out "{PF_RUN}/life_preds_seed123.jsonl"

In [ ]:
# Score every system variant (pilot numbers - plumbing validation only)
!python funnel_system/evaluate_system.py --run_dir "{PF_RUN}" --name PolitiFact++

## GossipCop++ — the citable result

~3,078-article shared test set per seed. The assemble cell picks `features_llama_multi` (all four
subsets) if it exists, else the binary `features_llama` (MF/MR only). With binary-only features,
human test articles have no LIFE features — fine for `oracle`/`two_bert`, and `funnel` falls back to
the human expert for any misrouted human article (counts reported). To close that gap properly, see
the **featurize misrouted articles** section at the bottom, then re-run assemble → LIFE → evaluate.

In [ ]:
!python funnel_system/make_split.py --data_dir "{GOSSIPCOP_DIR}" --out_dir "{GC_RUN}" --name GossipCop++

In [ ]:
# Router (RoBERTa AI-vs-Human, §7m config)
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed7.json" --role router --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed42.json" --role router --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed123.json" --role router --name GossipCop++ --out_dir "{GC_RUN}"

In [ ]:
# Human expert (plain BERT on HF/HR, §7o/§7q config, best-val)
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed7.json" --role human_expert --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed42.json" --role human_expert --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed123.json" --role human_expert --name GossipCop++ --out_dir "{GC_RUN}"

In [ ]:
# Machine-branch BERT control (plain BERT on MF/MR)
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed7.json" --role machine_bert --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed42.json" --role machine_bert --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed123.json" --role machine_bert --name GossipCop++ --out_dir "{GC_RUN}"

In [ ]:
# Monolithic control (one BERT on all four subsets, no routing)
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed7.json" --role monolithic --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed42.json" --role monolithic --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed123.json" --role monolithic --name GossipCop++ --out_dir "{GC_RUN}"

In [ ]:
# Assemble LIFE data. Prefers the 4-class features (full coverage); falls back to the
# binary MF/MR features. Re-running this cell also picks up on-demand misroute features.
import os
GC_FEATURES = GC_FEATURES_MULTI if os.path.isdir(GC_FEATURES_MULTI) else GC_FEATURES_BIN
GC_FEAT_ARGS = f'--features_dir "{GC_FEATURES}"'
if os.path.isdir(GC_MISROUTE_FEATURES) and os.listdir(GC_MISROUTE_FEATURES):
    GC_FEAT_ARGS += f' --features_dir "{GC_MISROUTE_FEATURES}"'
print('using:', GC_FEAT_ARGS)
!python funnel_system/assemble_life_data.py {GC_FEAT_ARGS} --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed7.json" --out_dir "{GC_RUN}"
!python funnel_system/assemble_life_data.py {GC_FEAT_ARGS} --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed42.json" --out_dir "{GC_RUN}"
!python funnel_system/assemble_life_data.py {GC_FEAT_ARGS} --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed123.json" --out_dir "{GC_RUN}"

In [ ]:
# LIFE branch per seed (see the PolitiFact note about per-epoch printed metrics)
!python LIFE_train/train_bce.py --train_path "{GC_RUN}/life_train_seed7.jsonl" --test_path "{GC_RUN}/life_test_seed7.jsonl" --num_train_epochs 50 --seed 7 --pred_out "{GC_RUN}/life_preds_seed7.jsonl"
!python LIFE_train/train_bce.py --train_path "{GC_RUN}/life_train_seed42.jsonl" --test_path "{GC_RUN}/life_test_seed42.jsonl" --num_train_epochs 50 --seed 42 --pred_out "{GC_RUN}/life_preds_seed42.jsonl"
!python LIFE_train/train_bce.py --train_path "{GC_RUN}/life_train_seed123.jsonl" --test_path "{GC_RUN}/life_test_seed123.jsonl" --num_train_epochs 50 --seed 123 --pred_out "{GC_RUN}/life_preds_seed123.jsonl"

In [ ]:
# The result table
!python funnel_system/evaluate_system.py --run_dir "{GC_RUN}" --name GossipCop++

## (Conditional) Featurize misrouted articles — GossipCop, binary-features-only case

Only needed if the evaluate cell above reported LIFE-missing fallbacks and you want the funnel's LIFE
branch to genuinely answer for misrouted human articles instead of falling back. This featurizes ONLY
the test articles that some router sent to LIFE without features (~a few hundred, minutes-to-an-hour):
Step 1 loads the existing GossipCop extractor checkpoint (`dataset/gossipcop/bert_bin.pt` — it skips
training when the file exists), then Steps 2–3 run the normal pipeline on the mini set.
**Afterwards: re-run the assemble → LIFE → evaluate cells above** (the assemble cell picks up the
extra features dir automatically).

In [ ]:
# Build the mini raw set: test articles routed to LIFE that lack features.
import json, os
from collections import defaultdict
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

needed = set()
for seed in (7, 42, 123):
    with open(f'{GC_RUN}/life_coverage_seed{seed}.json', encoding='utf-8') as f:
        missing = set(json.load(f)['missing_test_keys'])
    routed = set()
    with open(f'{GC_RUN}/preds_router_seed{seed}.jsonl', encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            if row['pred'] == 1:
                routed.add(row['key'])
    needed |= missing & routed
print(f'{len(needed)} unique test articles routed to LIFE without features (union over seeds)')

name_map = {'HF': ('HF_fake.jsonl', 'human_fake'), 'HR': ('HR_true.jsonl', 'human_true'),
            'MF': ('MF_fake.jsonl', 'gpt3.5_fake'), 'MR': ('MR_true.jsonl', 'gpt3.5_true')}
by_subset = defaultdict(list)
for key in needed:
    subset, idx = key.split(':')
    by_subset[subset].append(int(idx))

os.makedirs(GC_MISROUTE_RAW, exist_ok=True)
for subset, idxs in sorted(by_subset.items()):
    with open(f'{GOSSIPCOP_DIR}/{subset}.json', encoding='utf-8') as f:
        src = json.load(f)
    fname, label = name_map[subset]
    # sorted(idxs) keeps source order, so the new feature files stay an
    # ordered subsequence of their subset (required by the text join).
    with open(f'{GC_MISROUTE_RAW}/{fname}', 'w', encoding='utf-8') as f:
        for i in sorted(idxs):
            r = src[str(i)]
            f.write(json.dumps({'id': r['id'], 'text': r['text'], 'label': label},
                               ensure_ascii=False) + '\n')
    print(f'{subset}: {len(idxs)} articles -> {fname}')

In [ ]:
# Steps 1-3 on the mini set (extractor ckpt is LOADED, not retrained; top_k=15 = GossipCop's k)
!python dataset/1_keySentenceExtraction.py --data_dir "{GC_MISROUTE_RAW}" --output_file "{GC_MISROUTE_RAW}/key_sentences.jsonl" --top_k 15 --model_path "{PROJECT_DIR}/dataset/gossipcop/bert_bin.pt" --gpu 0
!python dataset/2_concate.py --folder_path "{GC_MISROUTE_RAW}" --important_sentences_file "{GC_MISROUTE_RAW}/key_sentences.jsonl"
!python dataset/3_gen_features_local.py --input_dir "{GC_MISROUTE_RAW}" --output_dir "{GC_MISROUTE_FEATURES}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16
# Now RE-RUN the GossipCop assemble -> LIFE -> evaluate cells above.

## Notes / caveats (read before quoting numbers)

- **Feature-space leak (inherited, flagged not fixed):** the Step-1 key-sentence extractor and the
  LLaMA features on Drive were generated ONCE over all articles, so LIFE's feature pipeline saw
  test-split articles. Only the classifiers (all five trained models) respect the split. A fully
  clean run would regenerate features per split — hours of A100 per seed; recorded as optional.
- **Group split shifts the anchors.** §7m/§7q used article-level splits; here a story and its
  GPT-3.5 rewrites travel together (id-groups), and the protocol is shared. Component anchors
  (router ~0.96 GC, human expert ~0.82 GC, LIFE ~0.87 PF) should land close but not identical —
  LARGE deviations mean a plumbing bug, not news.
- **LIFE protocol here vs §7e:** trains on the shared 70% (not its own 80/20 seed-0 split), and
  `--seed` matches the split seed. The per-epoch metrics train_bce.py prints are over the FULL mixed
  test set (including human articles LIFE never claims to handle) — ignore them; LIFE's own-slice
  anchor comes from evaluate_system.py.
- **Conventions:** veracity fake=1/real=0 everywhere in funnel_system (matches the LIFE BCE head;
  run_life_bert.py's HF-vs-HR track used the opposite — acc/macro-F1 comparisons are unaffected).
- **Label conflicts stay in the test set** (~0.8% of GossipCop bodies carry both labels, §7r): they
  are honest data noise, identical for every variant.
- `bce_en.pt` in the repo root is clobbered by every LIFE run — run cells sequentially, and don't
  run this notebook concurrently with the other LIFE notebooks.
- PolitiFact++ numbers are pilot/plumbing only (~85-article test, 1 article ≈ 1.2pp).